In [ ]:
from glob import glob
import os
import pandas as pd
import h5py
import numpy as np
import openslide
from PIL import Image
import matplotlib.pyplot as plt
import anndata as ad
def create_dir(path):
    if not os.path.exists(path):
        os.makedirs(path)
        
organ=['Ovary', 'Breast', 'Pancreas', 'Lymphoid', 'Prostate', 'Kidney',
       'Lung', 'Skin', 'Bowel', 'Heart', 'Brain', 'Bone', 'Liver',
       'Cervix', 'Lymph node', 'Bladder', 'Eye', 'Uterus']
st_technology=['Visium HD', "Visium HD 3'", 'Xenium', 'Visium',
       'Spatial Transcriptomics']
for o in organ:
    for t in st_technology:
        create_dir(f'../../data/marker_gene_spatial_expression/image/{t}/{o}')
        create_dir(f'../../data/marker_gene_spatial_expression/image_5x/{t}/{o}')
        create_dir(f'../../data/marker_gene_spatial_expression/label_proportion/{t}/{o}')
        create_dir(f'../../data/marker_gene_spatial_expression/label_intensity/{t}/{o}')
        
mpp=20
image_size=512

In [ ]:
data_path='../../data/spatialTranscriptome/'
meta_df = pd.read_csv(f'{data_path}meta_df_homo_sapiens.csv')

slide=openslide.open_slide(f'{data_path}wsis/{meta_df.loc[51]["image_filename"]}')
# h5ad 파일 읽기
adata = ad.read_h5ad(f'{data_path}st/{meta_df.loc[51]["id"]}.h5ad')

In [ ]:
import warnings
warnings.filterwarnings('ignore', message='Variable names are not unique')
from tqdm import tqdm

marker_genes = [
    'EPCAM', 'KRT8', 'KRT18', 'KRT19',
    'COL1A1', 'COL1A2', 'FAP', 'ACTA2',
    'CD3D', 'CD3E', 'CD8A',
    'MS4A1', 'CD79A',
    'CD68',
    'PECAM1'
]

# # ===== Pass 1: 전체 슬라이드에서 gene별 global percentile 계산 =====
# gene_values = {g: [] for g in marker_genes}
# valid_slide_ids = []

# for idx in tqdm(range(len(meta_df)), desc='Pass 1: Collecting gene stats'):
#     row = meta_df.iloc[idx]
#     sample_id = row['id']

#     h5ad_path = f'{data_path}st/{sample_id}.h5ad'
#     if not os.path.exists(h5ad_path):
#         continue

#     adata = ad.read_h5ad(h5ad_path)
#     adata.var_names_make_unique()
#     gene_names = adata.var_names.tolist()

#     missing = [g for g in marker_genes if g not in gene_names]
#     if len(missing) > 0:
#         continue

#     valid_slide_ids.append(sample_id)

#     gene_indices = [gene_names.index(g) for g in marker_genes]
#     if hasattr(adata.X, 'toarray'):
#         expr_matrix = adata.X[:, gene_indices].toarray()
#     else:
#         expr_matrix = np.array(adata.X[:, gene_indices])

#     for gi, gene in enumerate(marker_genes):
#         # 0이 아닌 발현값만 수집 (sparse 데이터 특성상 0이 대부분)
#         vals = expr_matrix[:, gi]
#         gene_values[gene].append(vals)

# # gene별 p1, p99 계산
# gene_p1 = {}
# gene_p99 = {}
# for gene in marker_genes:
#     all_vals = np.concatenate(gene_values[gene])
#     gene_p1[gene] = np.percentile(all_vals, 1)
#     gene_p99[gene] = np.percentile(all_vals, 99)
#     print(f'{gene:8s} | p1={gene_p1[gene]:.4f}, p99={gene_p99[gene]:.4f}')

# print(f'\nValid slides: {len(valid_slide_ids)} / {len(meta_df)}')

# # global stats 저장 (재사용 가능)
# global_stats = np.array([[gene_p1[g], gene_p99[g]] for g in marker_genes])
# np.save(f'{data_path}global_gene_stats_p1_p99.npy', global_stats)
# print(f'Saved: {data_path}global_gene_stats_p1_p99.npy')

gene_p1, gene_p99 = np.load(f'{data_path}global_gene_stats_p1_p99.npy').T
gene_stats = {g: (p1, p99) for g, p1, p99 in zip(marker_genes, gene_p1, gene_p99)}
print('Loaded global gene stats:')
for gene in marker_genes:
    p1, p99 = gene_stats[gene]
    print(f'{gene:8s} | p1={p1:.4f}, p99={p99:.4f}')

In [ ]:
from concurrent.futures import ThreadPoolExecutor

# ===== Pass 2: Grid 패치 추출 + 2-head label (proportion + intensity) =====

# global stats 로드
global_stats = np.load(f'{data_path}global_gene_stats_p1_p99.npy')  # (15, 2)
p99_arr = global_stats[:, 1].astype(np.float32)

target_mag = mpp  # 20 (target 20x)
context_mag = 5   # 5x context patch
patch_size = image_size  # 512
min_spots = 1  # 패치 내 최소 spot 수

save_image_dir = '../../data/marker_gene_spatial_expression/image'
save_image_5x_dir = '../../data/marker_gene_spatial_expression/image_5x'
save_proportion_dir = '../../data/marker_gene_spatial_expression/label_proportion'
save_intensity_dir = '../../data/marker_gene_spatial_expression/label_intensity'

# 이미 처리된 슬라이드 스킵 (재실행 시 이어서 처리)
import glob as glob_module
existing_files = set()
for f in glob_module.glob(f'{save_image_dir}/**/*.tiff', recursive=True):
    basename = os.path.splitext(os.path.basename(f))[0]
    parts = basename.rsplit('_', 2)
    if len(parts) == 3:
        existing_files.add(parts[0])

skipped = []
processed = []

for idx in tqdm(range(len(meta_df)), desc='Pass 2: Grid 2-head label extraction'):
    row = meta_df.iloc[idx]
    sample_id = row['id']
    organ_name = row['organ']
    st_tech = row['st_technology']
    image_file = row['image_filename']
    patch_count = 0

    if sample_id in existing_files:
        continue

    h5ad_path = f'{data_path}st/{sample_id}.h5ad'
    if not os.path.exists(h5ad_path):
        skipped.append((sample_id, 'h5ad not found'))
        continue

    adata = ad.read_h5ad(h5ad_path)
    adata.var_names_make_unique()
    gene_names = adata.var_names.tolist()

    missing = [g for g in marker_genes if g not in gene_names]
    if len(missing) > 0:
        skipped.append((sample_id, f'missing genes: {missing}'))
        continue

    wsi_path = f'{data_path}wsis/{image_file}'
    if not os.path.exists(wsi_path):
        skipped.append((sample_id, 'WSI not found'))
        continue

    slide = openslide.open_slide(wsi_path)
    slide_w, slide_h = slide.dimensions

    native_mag = float(str(row['magnification']).replace('x', '').replace('X', ''))
    downsample_20x = native_mag / target_mag
    downsample_5x = native_mag / context_mag

    full_patch_20x = int(patch_size * downsample_20x)
    full_patch_5x = int(patch_size * downsample_5x)

    target_w = int(slide_w / downsample_20x)
    target_h = int(slide_h / downsample_20x)
    n_patches_x = target_w // patch_size
    n_patches_y = target_h // patch_size

    if 'pxl_col_in_fullres' in adata.obs.columns and 'pxl_row_in_fullres' in adata.obs.columns:
        coords_x = adata.obs['pxl_col_in_fullres'].values.astype(float)
        coords_y = adata.obs['pxl_row_in_fullres'].values.astype(float)
    else:
        coords = adata.obsm['spatial']
        coords_x = coords[:, 0].astype(float)
        coords_y = coords[:, 1].astype(float)

    coords_20x_x = coords_x / downsample_20x
    coords_20x_y = coords_y / downsample_20x

    gene_indices = [gene_names.index(g) for g in marker_genes]
    if hasattr(adata.X, 'toarray'):
        expr_matrix = adata.X[:, gene_indices].toarray().astype(np.float32)
    else:
        expr_matrix = np.array(adata.X[:, gene_indices], dtype=np.float32)

    # CPM 정규화를 위한 spot별 total count
    if hasattr(adata.X, 'toarray'):
        total_counts = np.array(adata.X.sum(axis=1)).flatten().astype(np.float32)
    else:
        total_counts = adata.X.sum(axis=1).astype(np.float32)
    # counts per 10k
    cp10k = expr_matrix / (total_counts[:, None] + 1e-6) * 1e4

    def save_patch(px, py):
        patch_x0 = px * patch_size
        patch_y0 = py * patch_size

        in_patch = (
            (coords_20x_x >= patch_x0) & (coords_20x_x < patch_x0 + patch_size) &
            (coords_20x_y >= patch_y0) & (coords_20x_y < patch_y0 + patch_size)
        )
        n_in = in_patch.sum()
        if n_in < min_spots:
            return False

        patch_expr = expr_matrix[in_patch]  # (n_in, 15)
        patch_cp10k = cp10k[in_patch]       # (n_in, 15)

        # Head A: Proportion — gene별 양성 spot 비율 (expr > 0)
        label_prop = (patch_expr > 0).sum(axis=0).astype(np.float32) / n_in  # (15,)

        # Head B: Intensity — mean(log1p(counts_per_10k)), 양성 spot만
        label_int = np.zeros(15, dtype=np.float32)
        for gi in range(15):
            pos_mask = patch_expr[:, gi] > 0
            if pos_mask.sum() > 0:
                label_int[gi] = np.log1p(patch_cp10k[pos_mask, gi]).mean()
        # log1p(cp10k) max clipping (상한 ~10 정도, log1p(10000)≈9.21)
        label_int = np.clip(label_int, 0, 10.0)
        label_int = label_int / 10.0  # 0~1 정규화

        # 20x 패치 추출
        full_x = int(px * full_patch_20x)
        full_y = int(py * full_patch_20x)
        img_region = slide.read_region((full_x, full_y), 0, (full_patch_20x, full_patch_20x))
        img_patch = img_region.convert('RGB').resize((patch_size, patch_size), Image.LANCZOS)

        # 5x context 패치 (동일 중심, WSI 경계 clamp)
        center_x = full_x + full_patch_20x // 2
        center_y = full_y + full_patch_20x // 2
        half_5x = full_patch_5x // 2
        x5 = int(np.clip(center_x - half_5x, 0, max(0, slide_w - full_patch_5x)))
        y5 = int(np.clip(center_y - half_5x, 0, max(0, slide_h - full_patch_5x)))
        img_region_5x = slide.read_region((x5, y5), 0, (full_patch_5x, full_patch_5x))
        img_patch_5x = img_region_5x.convert('RGB').resize((patch_size, patch_size), Image.LANCZOS)

        patch_name = f'{sample_id}_{px}_{py}'
        img_patch.save(f'{save_image_dir}/{st_tech}/{organ_name}/{patch_name}.tiff')
        img_patch_5x.save(f'{save_image_5x_dir}/{st_tech}/{organ_name}/{patch_name}.tiff')
        np.save(f'{save_proportion_dir}/{st_tech}/{organ_name}/{patch_name}.npy', label_prop)
        np.save(f'{save_intensity_dir}/{st_tech}/{organ_name}/{patch_name}.npy', label_int)
        return True

    with ThreadPoolExecutor(max_workers=4) as executor:
        futures = []
        for py in range(n_patches_y):
            for px in range(n_patches_x):
                futures.append(executor.submit(save_patch, px, py))
        for f in futures:
            if f.result():
                patch_count += 1

    slide.close()
    processed.append((sample_id, patch_count))
    print(f'{sample_id} | {st_tech}/{organ_name} | {patch_count} patches saved')

print(f'\n=== Done ===')
print(f'Processed: {len(processed)} slides')
print(f'Skipped: {len(skipped)} slides')
for s_id, reason in skipped:
    print(f'  SKIP {s_id}: {reason}')

In [ ]:
# ===== 시각화: WSI 썸네일 + grid 패치 + 2-head label 오버레이 =====
from matplotlib.colors import Normalize
from matplotlib import cm

def visualize_slide(meta_row, data_path, marker_genes,
                    target_mag=20, patch_size=512, thumb_long_side=2000):
    """WSI 썸네일 위에 grid + proportion/intensity label 시각화"""
    sample_id = meta_row['id']
    image_file = meta_row['image_filename']

    h5ad_path = f'{data_path}st/{sample_id}.h5ad'
    wsi_path = f'{data_path}wsis/{image_file}'

    adata = ad.read_h5ad(h5ad_path)
    adata.var_names_make_unique()
    gene_names = adata.var_names.tolist()

    slide = openslide.open_slide(wsi_path)
    slide_w, slide_h = slide.dimensions

    native_mag = float(str(meta_row['magnification']).replace('x', '').replace('X', ''))
    downsample_20x = native_mag / target_mag

    # 썸네일
    aspect = slide_w / slide_h
    if aspect >= 1:
        thumb_w, thumb_h = thumb_long_side, int(thumb_long_side / aspect)
    else:
        thumb_w, thumb_h = int(thumb_long_side * aspect), thumb_long_side
    thumbnail = slide.get_thumbnail((thumb_w, thumb_h))
    thumb_w, thumb_h = thumbnail.size
    scale_x = thumb_w / slide_w
    scale_y = thumb_h / slide_h

    # spot 좌표
    if 'pxl_col_in_fullres' in adata.obs.columns and 'pxl_row_in_fullres' in adata.obs.columns:
        coords_x = adata.obs['pxl_col_in_fullres'].values.astype(float)
        coords_y = adata.obs['pxl_row_in_fullres'].values.astype(float)
    else:
        coords = adata.obsm['spatial']
        coords_x = coords[:, 0].astype(float)
        coords_y = coords[:, 1].astype(float)

    # gene expression
    gene_indices = [gene_names.index(g) for g in marker_genes]
    if hasattr(adata.X, 'toarray'):
        expr_matrix = adata.X[:, gene_indices].toarray().astype(np.float32)
    else:
        expr_matrix = np.array(adata.X[:, gene_indices], dtype=np.float32)

    # CPM: counts per 10k
    if hasattr(adata.X, 'toarray'):
        total_counts = np.array(adata.X.sum(axis=1)).flatten().astype(np.float32)
    else:
        total_counts = adata.X.sum(axis=1).astype(np.float32)
    cp10k = expr_matrix / (total_counts[:, None] + 1e-6) * 1e4

    # 20x grid 계산
    coords_20x_x = coords_x / downsample_20x
    coords_20x_y = coords_y / downsample_20x
    full_patch_20x = int(patch_size * downsample_20x)
    target_w = int(slide_w / downsample_20x)
    target_h = int(slide_h / downsample_20x)
    n_patches_x = target_w // patch_size
    n_patches_y = target_h // patch_size

    spot_size = max(2, min(30, int(thumb_long_side / 120)))

    # 패치별 정보 계산
    patch_info = {}
    for py in range(n_patches_y):
        for px in range(n_patches_x):
            patch_x0 = px * patch_size
            patch_y0 = py * patch_size
            in_patch = (
                (coords_20x_x >= patch_x0) & (coords_20x_x < patch_x0 + patch_size) &
                (coords_20x_y >= patch_y0) & (coords_20x_y < patch_y0 + patch_size)
            )
            cnt = int(in_patch.sum())
            if cnt >= 1:
                patch_info[(px, py)] = (cnt, in_patch)

    # 시각화할 gene 3개
    genes_to_show = ['EPCAM', 'CD68', 'COL1A1']
    cmaps_show = ['Reds', 'Blues', 'Greens']

    # ===== Figure: 3행 3열 =====
    fig, axes = plt.subplots(3, 3, figsize=(28, 26))

    # ── Row 1, Col 1: 썸네일 + grid + spot 위치 ──
    ax = axes[0, 0]
    ax.imshow(thumbnail)
    for px in range(n_patches_x + 1):
        ax.axvline(x=px * full_patch_20x * scale_x, color='cyan', linewidth=0.4, alpha=0.4)
    for py in range(n_patches_y + 1):
        ax.axhline(y=py * full_patch_20x * scale_y, color='cyan', linewidth=0.4, alpha=0.4)

    if patch_info:
        max_cnt = max(cnt for cnt, _ in patch_info.values())
        norm_cnt = Normalize(vmin=1, vmax=max(max_cnt, 2))
        cmap_patch = cm.YlGn
        for (px, py), (cnt, _) in patch_info.items():
            rx = px * full_patch_20x * scale_x
            ry = py * full_patch_20x * scale_y
            rw = full_patch_20x * scale_x
            rh = full_patch_20x * scale_y
            rect = plt.Rectangle((rx, ry), rw, rh,
                                 fill=True, facecolor=cmap_patch(norm_cnt(cnt)), alpha=0.3,
                                 edgecolor='lime', linewidth=0.8)
            ax.add_patch(rect)

    ax.scatter(coords_x * scale_x, coords_y * scale_y,
               s=spot_size, c='red', alpha=0.6, edgecolors='darkred', linewidths=0.3)
    ax.set_title(f'Grid + Spots\n{n_patches_x}x{n_patches_y} grid | '
                 f'{len(patch_info)} valid patches | {len(coords_x)} spots',
                 fontsize=12, fontweight='bold')
    ax.axis('off')

    # ── Row 1, Col 2: 패치별 spot 수 heatmap ──
    ax = axes[0, 1]
    ax.imshow(thumbnail, alpha=0.4)
    if patch_info:
        for (px, py), (cnt, _) in patch_info.items():
            rx = px * full_patch_20x * scale_x
            ry = py * full_patch_20x * scale_y
            rw = full_patch_20x * scale_x
            rh = full_patch_20x * scale_y
            rect = plt.Rectangle((rx, ry), rw, rh,
                                 fill=True, facecolor=cmap_patch(norm_cnt(cnt)), alpha=0.7,
                                 edgecolor='gray', linewidth=0.3)
            ax.add_patch(rect)
            if rw > 25 and rh > 25:
                ax.text(rx + rw / 2, ry + rh / 2, str(cnt),
                        ha='center', va='center', fontsize=6, color='black', fontweight='bold')
        sm = cm.ScalarMappable(cmap=cmap_patch, norm=norm_cnt)
        sm.set_array([])
        plt.colorbar(sm, ax=ax, fraction=0.046, pad=0.04, label='spots per patch')
    ax.set_title('Spots per Patch', fontsize=12, fontweight='bold')
    ax.axis('off')

    # ── Row 1, Col 3: 비워두기 (or 전체 통계) ──
    ax = axes[0, 2]
    ax.axis('off')
    if patch_info:
        spot_counts = [cnt for cnt, _ in patch_info.values()]
        stats_text = (
            f'Total spots: {len(coords_x)}\n'
            f'Valid patches: {len(patch_info)} / {n_patches_x * n_patches_y}\n'
            f'Spots/patch: min={min(spot_counts)}, med={int(np.median(spot_counts))}, '
            f'max={max(spot_counts)}, mean={np.mean(spot_counts):.1f}\n\n'
            f'Technology: {meta_row["st_technology"]}\n'
            f'Organ: {meta_row["organ"]}\n'
            f'Magnification: {meta_row["magnification"]}'
        )
        ax.text(0.1, 0.5, stats_text, transform=ax.transAxes,
                fontsize=14, verticalalignment='center', fontfamily='monospace',
                bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))
        ax.set_title('Summary', fontsize=12, fontweight='bold')

    # ── Row 2: Head A — Proportion (gene별 양성 비율) ──
    norm_prop = Normalize(vmin=0, vmax=1)
    for col, (gene, cmap_name) in enumerate(zip(genes_to_show, cmaps_show)):
        ax = axes[1, col]
        ax.imshow(thumbnail, alpha=0.4)
        gi = marker_genes.index(gene)
        for (px, py), (cnt, in_patch) in patch_info.items():
            prop = (expr_matrix[in_patch, gi] > 0).sum() / cnt
            rx = px * full_patch_20x * scale_x
            ry = py * full_patch_20x * scale_y
            rw = full_patch_20x * scale_x
            rh = full_patch_20x * scale_y
            color = cm.get_cmap(cmap_name)(norm_prop(prop))
            rect = plt.Rectangle((rx, ry), rw, rh,
                                 fill=True, facecolor=color, alpha=0.7,
                                 edgecolor='gray', linewidth=0.3)
            ax.add_patch(rect)
        sm = cm.ScalarMappable(cmap=cmap_name, norm=norm_prop)
        sm.set_array([])
        plt.colorbar(sm, ax=ax, fraction=0.046, pad=0.04, label='proportion')
        ax.set_title(f'{gene} — Proportion (Head A)', fontsize=12, fontweight='bold')
        ax.axis('off')

    # ── Row 3: Head B — Intensity (mean log1p(cp10k), 양성 spot만) ──
    norm_int = Normalize(vmin=0, vmax=1)
    for col, (gene, cmap_name) in enumerate(zip(genes_to_show, cmaps_show)):
        ax = axes[2, col]
        ax.imshow(thumbnail, alpha=0.4)
        gi = marker_genes.index(gene)
        for (px, py), (cnt, in_patch) in patch_info.items():
            pos_mask = expr_matrix[in_patch, gi] > 0
            if pos_mask.sum() > 0:
                intensity = np.clip(np.log1p(cp10k[in_patch][pos_mask, gi]).mean() / 10.0, 0, 1)
            else:
                intensity = 0.0
            rx = px * full_patch_20x * scale_x
            ry = py * full_patch_20x * scale_y
            rw = full_patch_20x * scale_x
            rh = full_patch_20x * scale_y
            color = cm.get_cmap(cmap_name)(norm_int(intensity))
            rect = plt.Rectangle((rx, ry), rw, rh,
                                 fill=True, facecolor=color, alpha=0.7,
                                 edgecolor='gray', linewidth=0.3)
            ax.add_patch(rect)
        sm = cm.ScalarMappable(cmap=cmap_name, norm=norm_int)
        sm.set_array([])
        plt.colorbar(sm, ax=ax, fraction=0.046, pad=0.04, label='intensity (norm)')
        ax.set_title(f'{gene} — Intensity (Head B)', fontsize=12, fontweight='bold')
        ax.axis('off')

    plt.suptitle(f'{sample_id}  |  {meta_row["st_technology"]} / {meta_row["organ"]}',
                 fontsize=16, fontweight='bold', y=1.01)
    plt.tight_layout()
    plt.show()
    slide.close()

    # 통계 출력
    if patch_info:
        spot_counts = [cnt for cnt, _ in patch_info.values()]
        print(f'Total spots: {len(coords_x)}')
        print(f'Valid patches: {len(patch_info)} / {n_patches_x * n_patches_y} '
              f'({100 * len(patch_info) / max(1, n_patches_x * n_patches_y):.1f}%)')
        print(f'Spots per patch: min={min(spot_counts)}, median={int(np.median(spot_counts))}, '
              f'max={max(spot_counts)}, mean={np.mean(spot_counts):.1f}')


# 샘플 슬라이드 시각화
visualize_slide(meta_df.iloc[2], data_path, marker_genes)